In [ ]:
from graphviz import Digraph
from IPython.display import Image, display

# Extract schema from SQLite via DBManager
with DBManager() as db:
	db.execute_query("""
		SELECT name
		FROM sqlite_master
		WHERE type='table' AND name NOT LIKE 'sqlite_%'
		ORDER BY name
	""")
	tables = [row[0] for row in db.cursor.fetchall()]

	schema = {}
	relations = []

	for table in tables:
		db.execute_query(f"PRAGMA table_info({table})")
		cols = db.cursor.fetchall()  # cid, name, type, notnull, dflt_value, pk
		schema[table] = cols

		db.execute_query(f"PRAGMA foreign_key_list({table})")
		fks = db.cursor.fetchall()  # id, seq, table, from, to, on_update, on_delete, match
		for fk in fks:
			relations.append((table, fk[3], fk[2], fk[4]))  # from_table, from_col, to_table, to_col

# Build diagram
dot = Digraph("database_schema", format="png")
dot.attr(rankdir="LR")

for table, cols in schema.items():
	col_lines = []
	for _, name, col_type, notnull, _, pk in cols:
		flags = []
		if pk:
			flags.append("PK")
		if notnull:
			flags.append("NOT NULL")
		suffix = f" ({', '.join(flags)})" if flags else ""
		col_lines.append(f"{name}: {col_type}{suffix}\\l")
	label = f"{{{table}|{''.join(col_lines)}}}"
	dot.node(table, label=label, shape="record")

for from_table, from_col, to_table, to_col in relations:
	dot.edge(from_table, to_table, label=f"{from_col} -> {to_col}")

# Export image
output_file = dot.render("db_schema", cleanup=True)
print(f"Schema image exported to: {output_file}")

# Preview in notebook
display(Image(filename=output_file))

In [ ]:
paire_id = 'JOMR'
import pandas as pd

from db_manager import DBManager
with DBManager() as db:

    subquery = f"""
        SELECT tp.point_id, tg.game_id, tp.team_a_score, tp.team_a_score_diff, tg.team_a, tg.team_b
        FROM table_game AS tg
        LEFT JOIN table_point AS tp ON tg.game_id = tp.game_id
        WHERE tg.team_a = '{paire_id}'"""

    db.execute_query(f"""SELECT ts.point_id, ts.player, ts.grade, ts.point_won, sub.game_id, sub.team_a_score, sub.team_a_score_diff, sub.team_b
                     FROM table_serve AS ts
                     LEFT JOIN ({subquery}) AS sub
                     ON ts.point_id = sub.point_id
                     WHERE ts.paire_id = '{paire_id}'
                     """)

    results = db.cursor.fetchall()

results_df = pd.DataFrame(results, columns=[desc[0] for desc in db.cursor.description])
results_df.head()

✅ Connection established: c:\Users\habib\Documents\GitHub\DataBeach\database\databeach_base.db
🔒 Connection closed.


,point_id,player,grade,point_won,game_id,team_a_score,team_a_score_diff,team_b
0,JOMR_jan26_MBV_01_p2,RANC Mathilde,out-of-system pass,0,JOMR_jan26_MBV_01,1,1,NoG_AliP
1,JOMR_jan26_MBV_01_p5,OFFREDI Jade,error,0,JOMR_jan26_MBV_01,2,0,NoG_AliP
2,JOMR_jan26_MBV_01_p7,RANC Mathilde,ace,1,JOMR_jan26_MBV_01,3,0,NoG_AliP
3,JOMR_jan26_MBV_01_p8,RANC Mathilde,ace,1,JOMR_jan26_MBV_01,4,1,NoG_AliP
4,JOMR_jan26_MBV_01_p9,RANC Mathilde,ace,1,JOMR_jan26_MBV_01,5,2,NoG_AliP


: 

In [5]:
from db_manager import DBManager
with DBManager() as db:
    db.execute_query(f"""SELECT point_id 
                     FROM table_serve 
                     WHERE grade = 'undetermined'
                     """)
    results = db.cursor.fetchall()

undetermined_points = [row[0] for row in results]
len(undetermined_points)

segmented_point_dir = r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs preprocess\points_segmented'

import os
import shutil
os.makedirs(f'{segmented_point_dir}\\echange Thib-Jade', exist_ok=True)
for point_id in undetermined_points:
    # Build the full path of the source video file for the current point
    src = f'{segmented_point_dir}\\{point_id}.mp4'
    
    # Build the destination path inside the "echange Thib-Jade" subfolder
    dst = f'{segmented_point_dir}\\echange Thib-Jade\\{point_id}.mp4'
    
    # Copy the file only if it exists at the source location
    if os.path.exists(src):
        shutil.copy2(src, dst)



✅ Connection established: c:\Users\habib\Documents\GitHub\DataBeach\database\databeach_base.db
🔒 Connection closed.


In [ ]:
import pandas as pd
indexed_df_points_csv_path = r'C:\Users\habib\Documents\GitHub\DataBeach\indexed_df_points\indexed_df_points_JOMR_jan26_MBV_02.csv'
indexed_df = pd.read_csv(indexed_df_points_csv_path)
# type(indexed_df['JOMR_score'])
# indexed_df['JOMR_score']
indexed_df.head()
indexed_df.iloc[:, 3]

0      0
1      1
2      2
3      2
4      2
5      3
6      3
7      4
8      4
9      4
10     4
11     4
12     5
13     5
14     5
15     5
16     6
17     7
18     7
19     8
20     9
21     9
22     9
23    10
24    10
25    10
26    11
27    12
28    12
29    13
30    13
31    13
32    14
33    14
34    15
35    16
36    16
37    16
38    16
39    16
40    17
41    18
Name: JOMR_score, dtype: int64

In [48]:
from game_editor import GameEditor
editor = GameEditor(
    # video_path=r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs bruts\AleD-RonP_mar26_session_01.mp4',
    video_dir=r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs bruts',
    output_dir=r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs preprocess'
)

In [49]:
editor.pre_match_editing()


Processing video: C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs bruts\AleD-RonP_mar26_session_01.mp4
Last frame index: 47394
Match start marked at frame 289, i.e. 9.64 seconds


In [52]:
editor = GameEditor(
    video_path=r'C:\Users\habib\Desktop\Montages volley et beach\Alex&Co\AleD-RonP_mar26_session_01_started.mp4',
    output_dir=r'C:\Users\habib\Desktop\Montages volley et beach\Alex&Co'
)

editor.game_to_segmented_points(
    team1_name='AleD_RonP',
    team2_name='MarB_JerD',
    rewrite_videos=True,
)

Possible inconsistency detected: scores at switch time are not multiples of 5 or 7.
Scores at switch time: [np.int64(14), np.int64(21), np.int64(26)]
Cutting point 1: frames 76 - 372
Cutting point 2: frames 924 - 1024
Cutting point 3: frames 1421 - 1817
Cutting point 4: frames 2424 - 2654
Cutting point 5: frames 3098 - 3350
Cutting point 6: frames 3767 - 4008
Cutting point 7: frames 4306 - 4715
Cutting point 8: frames 5289 - 5384
Cutting point 9: frames 5775 - 6400
Cutting point 10: frames 6844 - 7046
Cutting point 11: frames 7366 - 7501
Cutting point 12: frames 7986 - 8345
Cutting point 13: frames 8687 - 9152
Cutting point 14: frames 9709 - 9894
Cutting point 15: frames 10521 - 10912
Cutting point 16: frames 11319 - 11579
Cutting point 17: frames 11888 - 12236
Cutting point 18: frames 12747 - 13036
Cutting point 19: frames 13592 - 13799
Cutting point 20: frames 14276 - 14406
Cutting point 21: frames 14810 - 15114
Cutting point 22: frames 15881 - 16211
Cutting point 23: frames 16656 - 

In [23]:
from video_edit_utils import *
import pandas as pd
import os

action_df = pd.read_csv(r'C:\Users\habib\Documents\GitHub\DataBeach\scripts\indexed_df_points\indexed_df_points_AleD-RonP_mar26_session_01_started.csv')

action_df.head()


,point_index,service_side,start_frame,end_frame,AleD_RonP_score,MarB_JerD_score,AleD_RonP_sets,MarB_JerD_sets
0,1,match start - service AleD_RonP,76,372,0,0,0,0
1,2,service MarB_JerD,924,1024,0,1,0,0
2,3,service MarB_JerD,1421,1817,0,2,0,0
3,4,service AleD_RonP,2424,2654,1,2,0,0
4,5,service AleD_RonP,3098,3350,2,2,0,0


In [24]:
extract_segments_from_df_gpu(
    video_path = r'C:\Users\habib\Desktop\Montages volley et beach\Alex&Co\preprocessed games\AleD-RonP_vs_JerM_mar26_session_01_started.mp4',
    actions_df=action_df,
    output_dir=r'C:\Users\habib\Desktop\Montages volley et beach\Alex&Co\segmented points'
)

Cutting point 1: frames 76 - 372


CalledProcessError: Command '['C:\\ffmpeg\\bin\\ffmpeg.exe', '-y', '-hwaccel', 'cuda', '-i', 'C:\\Users\\habib\\Desktop\\Montages volley et beach\\Alex&Co\\preprocessed games\\AleD-RonP_vs_JerM_mar26_session_01_started.mp4', '-vf', "select='between(n,76,372)',setpts=PTS-STARTPTS", '-an', '-c:v', 'h264_nvenc', '-preset', 'p4', '-rc', 'constqp', '-qp', '18', 'C:\\Users\\habib\\Desktop\\Montages volley et beach\\Alex&Co\\segmented points\\AleD-RonP_vs_JerM_mar26_session_01_started_p1.mp4']' returned non-zero exit status 4294967294.

In [3]:
import pandas as pd
test_indexed_df = pd.read_csv(
    r'C:\Users\habib\Documents\GitHub\DataBeach\indexed_df_points\indexed_df_points_JOMR_jan26_MBV_01.csv'
)
test_indexed_df.drop(columns=['point_index'], inplace=True)
test_indexed_df.head()
df = test_indexed_df
df.head()

,service_side,start_frame,end_frame,JOMR_score,NoG_AliP_score,JOMR_sets,NoG_AliP_sets
0,match start - service NoG_AliP,134,440,0,0,0,0
1,service JOMR,749,1168,1,0,0,0
2,service NoG_AliP,1453,1881,1,1,0,0
3,service NoG_AliP,2156,2546,1,2,0,0
4,service JOMR,2903,3065,2,2,0,0


In [10]:
df.to_csv(path_or_buf=r'C:\Users\habib\Documents\GitHub\DataBeach\scripts\test_indexed_df.csv', index=False)
new_indexed_df = pd.read_csv(r'C:\Users\habib\Documents\GitHub\DataBeach\scripts\test_indexed_df.csv')
new_indexed_df.head()


,service_side,start_frame,end_frame,JOMR_score,NoG_AliP_score,JOMR_sets,NoG_AliP_sets,point_index
0,match start - service NoG_AliP,134,440,0,0,0,0,1
1,service JOMR,749,1168,1,0,0,0,2
2,service NoG_AliP,1453,1881,1,1,0,0,3
3,service NoG_AliP,2156,2546,1,2,0,0,4
4,service JOMR,2903,3065,2,2,0,0,5


In [1]:
from game_editor import GameEditor
editor = GameEditor(
    video_path=r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs preprocess\JOMR_mar26_VSG_01_started_rotated_270.mp4',
    output_dir=r'C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs preprocess\points_segmented'
)

In [2]:
editor.game_to_segmented_points(
    team1_name="JOMR",
    team2_name="(adv)",
    rewrite_videos=True
)

game_id: JOMR_mar26_VSG_01
Cutting point 001: frames 70 - 567
video_path: C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs preprocess\JOMR_mar26_VSG_01_started_rotated_270.mp4
start_frame: 70
end_frame: 567
output_dir: C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs preprocess\points_segmented
output_video: C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs preprocess\points_segmented\JOMR_mar26_VSG_01_p001.mp4
Cutting point 002: frames 842 - 1168
video_path: C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs preprocess\JOMR_mar26_VSG_01_started_rotated_270.mp4
start_frame: 842
end_frame: 1168
output_dir: C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs preprocess\points_segmented
output_video: C:\Users\habib\Desktop\Montages volley et beach\Jade&Math\matchs preprocess\points_segmented\JOMR_mar26_VSG_01_p002.mp4
Cutting point 003: frames 1528 - 1699
video_path: C:\Users\habib\Desktop\Montages volley et beach\J

In [14]:
import os
indexed_df_dir = r'C:\Users\habib\Documents\GitHub\DataBeach\indexed_df_points'
indexed_df_csv = [f for f in os.listdir(indexed_df_dir) if f.endswith('.csv')]
indexed_df_csv.remove('indexed_df_points_AleD-RonP_mar26_session_01.csv')
indexed_df_csv.remove('indexed_df_points_JOMR_mar26_VSG_01.csv')
# indexed_df_csv

# for f in indexed_df_csv:


    # point_number = str(f.split('_p')[1].split('.')[0])
    # if len(point_number) != 3:
    #     if len(point_number) == 1:
    #         new_point_number = '00' + point_number
    #     elif len(point_number) == 2:
    #         new_point_number = '0' + point_number


In [ ]:
import os
import pandas as pd
csv_file = r'C:\Users\habib\Documents\GitHub\DataBeach\indexed_df_points\(test) indexed_df_points_JOMR_jan26_MBV_01.csv'
df = pd.read_csv(csv_file)
df['point_index'] = df['point_index'].apply(lambda x: str('999') if pd.isnull(x) else x)
df['point_index'] = df['point_index'].apply(lambda x: int(float(x)) if x != '999' else x)
# df['point_index'] = df['point_index'].apply(lambda x: x.zfill(3) if x != '999' else x)
df.head(10)

,service_side,start_frame,end_frame,JOMR_score,NoG_AliP_score,JOMR_sets,NoG_AliP_sets,point_index
0,match start - service NoG_AliP,134,440,0,0,0,0,1.0
1,service JOMR,749,1168,1,0,0,0,2.0
2,service NoG_AliP,1453,1881,1,1,0,0,3.0
3,service NoG_AliP,2156,2546,1,2,0,0,4.0
4,service JOMR,2903,3065,2,2,0,0,5.0
5,service NoG_AliP,3420,3858,2,3,0,0,6.0
6,service JOMR,4250,4398,3,3,0,0,7.0
7,*SWITCH*,4574,4894,3,3,0,0,999
8,service JOMR,5054,5210,4,3,0,0,8.0
9,service JOMR,5598,5854,5,3,0,0,9.0
